# 01 — Data Quality

Loads the example experiment's market data through ``DataLoader``, which uses
the local cache and downloads missing ranges when the source is available.
It then runs the ``DataValidator`` and inspects coverage and basic statistics
before any research begins.

This step exists because a backtest is only as trustworthy as its input data: duplicates, gaps, and impossible OHLC bars must be
caught before they silently distort results.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from quantlab.config import ExperimentConfig
from quantlab.data.loader import DataLoader
from quantlab.data.base import price_matrix

config = ExperimentConfig.from_yaml("../configs/momentum_sp500.yaml")
config.experiment_name, config.symbols, config.start_date, config.end_date


In [ ]:
data, report = DataLoader().load(config)
print(report.summary())
for w in report.warnings:
    print(" -", w)


## Coverage per symbol

In [ ]:
data.groupby("symbol")["timestamp"].agg(["min", "max", "count"])


## Adjusted close price series

In [ ]:
prices = price_matrix(data)
fig, ax = plt.subplots(figsize=(10, 5))
(prices / prices.iloc[0] * 100).plot(ax=ax, lw=1.2)
ax.set_title("Rebased adjusted close (100 = start of sample)")
ax.set_ylabel("Index level")
ax.legend(ncol=4, fontsize=8)
plt.show()


## Daily return summary statistics

In [ ]:
returns = prices.pct_change(fill_method=None)
returns.describe().T[["mean", "std", "min", "max"]].style.format("{:.4f}")


## Takeaways

- No duplicate `(timestamp, symbol)` rows and no non-positive prices were
  found (or they would appear as warnings above).
- Coverage spans the full requested range for every symbol.
- Returns look like ordinary daily equity/ETF returns (no obviously broken
  bars). This is the baseline the rest of the research builds on.
